# 02 - Scalar Graph

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 怎么学这本

建议顺序：

```text
先运行代码 -> 看输出 -> 回到公式手算 -> 再看输出是否一致
```

本节过关标准：

```text
1. 能说出 a、b、c、d、L 的 data 是多少
2. 能说出 backward 前 grad 为什么都是 0
3. 能说出 backward 后 a.grad=8、b.grad=4 分别代表什么
4. 能把 a.grad 和 dL/da 对上，把 b.grad 和 dL/db 对上
```

暂时不用研究内部实现。先把这一节当成一次实验：输入公式，观察前向结果和反向梯度。


## 本节不要研究源码

这一节只练使用层心智模型：

```text
Value 数字 -> 前向算出 L.data -> L.backward() -> 看 a.grad / b.grad
```

看到 `_prev`、`_op`、`_backward` 时，只把它们当成观察信息。真正源码解释在下一节。

## 1. Forward: Build A Tiny Formula

把公式拆成几步：

$$
c = ab
$$

$$
d = c + a
$$

$$
L = 2d
$$

对应代码：


### Predict Before Run

先别运行下一格。你先写下这 5 个值：

```text
a.data = ?
b.data = ?
c.data = a*b = ?
d.data = c+a = ?
L.data = d*2 = ?
```

写完再运行。

In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a * b
d = c + a
L = d * 2

print('a =', a)
print('b =', b)
print('c = a * b =', c)
print('d = c + a =', d)
print('L = d * 2 =', L)


你现在先只看 `data`：

```text
a.data = 2
b.data = 3
c.data = 6
 d.data = 8
L.data = 16
```

这就是前向计算。

注意：这里的 `Value(data=6.0, grad=0)` 表示这个节点当前数值是 6，梯度还没计算，所以 `grad=0`。


## 2. A Small Printer For Nodes

下面这个 `show_all` 只是为了把每个节点打印得更清楚。

你不用纠结它的 Python 写法，只需要看输出里的四列：

```text
name     节点名字
data     当前数值
grad     当前梯度
parents  这个节点由哪些节点算出来
```


In [ ]:
nodes = {'a': a, 'b': b, 'c': c, 'd': d, 'L': L}


def value_name(value, nodes):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'Value({value.data})'


def show(name, value, nodes):
    parents = [value_name(parent, nodes) for parent in value._prev]
    print(f'{name:>2} | data={value.data:>5} | grad={value.grad:>5} | parents={parents}')


def show_all(title, nodes):
    print(title)
    for name in ['a', 'b', 'c', 'd', 'L']:
        show(name, nodes[name], nodes)


show_all('Before backward', nodes)


观察输出：

```text
a, b 是最初给的数，所以 parents=[]
c 来自 a 和 b
d 来自 c 和 a
L 来自 d 和普通数字 2
```

这就是“计算图”的意思：每个中间结果都能追溯自己从哪里来。

这本只要求你能看懂这个现象；至于它源码怎么记录，下一节再学。


## 3. Backward: Ask How L Changes With a And b

现在调用：

```python
L.backward()
```

它会自动计算：

$$
\frac{\partial L}{\partial a}
$$

和：

$$
\frac{\partial L}{\partial b}
$$

也就是：

```text
a.grad
b.grad
```


In [ ]:
L.backward()
show_all('After backward', nodes)


重点看最后：

```text
a.grad = 8
b.grad = 4
```

含义是：

```text
a 稍微变大一点，L 大约按 8 倍速度变化
b 稍微变大一点，L 大约按 4 倍速度变化
```

也就是：

$$
a.grad = \frac{\partial L}{\partial a}
$$

$$
b.grad = \frac{\partial L}{\partial b}
$$


## 4. Check With Hand Math

原式：

$$
L = 2(ab+a)
$$

对 $a$ 求偏导，把 $b$ 当常数：

$$
\frac{\partial L}{\partial a} = 2(b+1)
$$

对 $b$ 求偏导，把 $a$ 当常数：

$$
\frac{\partial L}{\partial b} = 2a
$$

代入 $a=2,b=3$：

$$
\frac{\partial L}{\partial a}=2(3+1)=8
$$

$$
\frac{\partial L}{\partial b}=2\times2=4
$$

和 `micrograd` 输出一致。


In [ ]:
expected_da = 2 * (3 + 1)
expected_db = 2 * 2

print('hand math dL/da =', expected_da)
print('micrograd a.grad =', a.grad)
print()
print('hand math dL/db =', expected_db)
print('micrograd b.grad =', b.grad)


## 5. Numerical Check

再用“微小扰动”验证一下。

如果把 $a$ 稍微加一点点，看看 $L$ 增加多少，也能近似得到导数：

$$
\frac{\partial L}{\partial a}
\approx
\frac{f(a+h,b)-f(a-h,b)}{2h}
$$


In [ ]:
def f(a_value, b_value):
    return 2 * (a_value * b_value + a_value)

h = 1e-6
num_da = (f(2.0 + h, 3.0) - f(2.0 - h, 3.0)) / (2 * h)
num_db = (f(2.0, 3.0 + h) - f(2.0, 3.0 - h)) / (2 * h)

print('numerical dL/da =', num_da)
print('micrograd a.grad =', a.grad)
print()
print('numerical dL/db =', num_db)
print('micrograd b.grad =', b.grad)


## 6. Mini Lab: 改一行，重新跑一遍

下面把同一个公式封装成函数。你可以改 `examples` 里的数字，观察 `data` 和 `grad` 怎么变。

In [ ]:
def run_value_example(a_data, b_data):
    a = Value(a_data)
    b = Value(b_data)
    L = 2 * (a * b + a)
    L.backward()
    expected_da = 2 * (b_data + 1)
    expected_db = 2 * a_data
    print(f'a={a_data:>5}, b={b_data:>5}, L.data={L.data:>6}, a.grad={a.grad:>6}, b.grad={b.grad:>6}')
    assert_close(a.grad, expected_da, 'a.grad')
    assert_close(b.grad, expected_db, 'b.grad')
    print()

examples = [(2.0, 3.0), (4.0, -2.0), (0.0, 5.0)]
for a_data, b_data in examples:
    run_value_example(a_data, b_data)

## What To Remember

- 跑通主线。
- 说清楚本节的输入、输出、梯度或训练含义。
- 完成底部课堂检查后再进入 homework。

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - 前向 data

In [ ]:
# a=2,b=3,c=a*b,d=c+a,L=d*2
predict_values = None  # [a,b,c,d,L]



qa_check('qa_check_class_02_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先分清 `data` 和 `grad`：`data` 前向就有，`grad` 要在 `backward()` 后才有；重复 backward 会累加。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `predict_values` 填 `[2, 3, 6, 8, 16]`。

</details>

## Run - 真的跑 Value

In [ ]:
from micrograd.engine import Value

a = Value(2.0)
b = Value(3.0)
c = a*b
d = c+a
L = d*2
print('before:', a, b, c, d, L)
L.backward()
print('after:', a, b, c, d, L)

## Modify - 改一个输入

In [ ]:
# 改成 a=4,b=3。预测 L.data/a.grad/b.grad
student_L_data = None
student_a_grad = None
student_b_grad = None



qa_check('qa_check_class_02_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先分清 `data` 和 `grad`：`data` 前向就有，`grad` 要在 `backward()` 后才有；重复 backward 会累加。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_L_data` 填 `L.data`。
- `student_a_grad` 填 `a.grad`。
- `student_b_grad` 填 `b.grad`。

</details>

## 接回主线

`backward()` 后，参数的 `grad` 才能指导下一步往哪里更新。